In [4]:
# =========================
# CONVERTING HO TO YEO
# =========================
import os, re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

BEDFORD_XLSX    = os.path.join(BASE_DIR, "Bedford_2023_significant_edges_FINAL.xlsx")
SMIGIELSKI_XLSX = os.path.join(BASE_DIR, "Smigielski_2019_significant_edges_FINAL.xlsx")  # <-- same naming convention

HO_NII  = os.path.join(ATLAS_DIR, "conn_atlas_harvardoxford.nii")
HO_TXT  = os.path.join(ATLAS_DIR, "conn_atlas_harvardoxford.txt")
YEO_NII = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR = os.path.join(BASE_DIR, "data_yeo_converted")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_BEDFORD    = os.path.join(OUT_DIR, "Bedford_2023_Yeo7_converted.xlsx")
OUT_SMIGIELSKI = os.path.join(OUT_DIR, "Smigielski_2019_Yeo7_converted.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def load_ho_label_to_id(txt_path):
    labels = []
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                labels.append(clean_ws(ln))
    return {lab: i+1 for i, lab in enumerate(labels)}  # 1..N

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a); nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def compute_ho_to_yeo7(ho_nii_path, yeo_nii_path):
    ho_img  = nib.load(ho_nii_path)
    yeo_img = nib.load(yeo_nii_path)

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, ho_img, interpolation="nearest")
    ho  = ho_img.get_fdata().astype(int)
    yeo = yeo_rs.get_fdata().astype(int)

    ho_ids  = [i for i in np.unique(ho) if i != 0]
    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    ho_to_yeo = {}
    for hid in ho_ids:
        hm = (ho == hid)
        best_name, best_d, best_inter = None, -1.0, -1
        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(hm & ym)
            d = dice(hm, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)
        ho_to_yeo[hid] = best_name
    return ho_to_yeo

def require_map(label, ho_label_to_id, ho_to_yeo, row_idx, colname, study_tag):
    raw = label
    lab = clean_ws(raw)

    if lab not in ho_label_to_id:
        hints = [k for k in ho_label_to_id.keys() if lab.lower() in k.lower() or k.lower() in lab.lower()]
        hint_txt = ""
        if hints:
            hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints[:10])
        raise ValueError(
            f"[{study_tag}] Row {row_idx}: '{colname}' not found in Harvard-Oxford labels.\n"
            f"  Raw value: {raw!r}\n  Cleaned: {lab!r}{hint_txt}"
        )

    hid = ho_label_to_id[lab]
    yeo = ho_to_yeo.get(hid, None)
    if yeo not in YEO7:
        raise ValueError(
            f"[{study_tag}] Row {row_idx}: '{colname}' mapped HO id {hid} but got invalid Yeo-7 value: {yeo!r}\n"
            f"  Raw value: {raw!r}\n  Cleaned: {lab!r}"
        )
    return yeo

def convert_file_to_yeo7(xlsx_path, out_xlsx, study_name):
    df = pd.read_excel(xlsx_path)
    df.columns = df.columns.astype(str).str.replace("\u00a0"," ", regex=False).str.strip()

    seed_c, targ_c, dir_c, net_c, con_c = "Seed", "Target", "Direction", "Network", "Contrast"

    # Keep only Harvard-Oxford rows
    sub = df[df[net_c].astype(str).str.strip().str.lower().eq("harvard-oxford")].copy()

    seed_yeo, targ_yeo = [], []
    for i, r in sub.iterrows():
        seed_yeo.append(require_map(r[seed_c], ho_label_to_id, ho_to_yeo, i, "Seed", study_name))
        targ_yeo.append(require_map(r[targ_c], ho_label_to_id, ho_to_yeo, i, "Target", study_name))

    out = pd.DataFrame({
        "Study": study_name,
        "Contrast": sub[con_c].astype(str).values,
        "Seed": seed_yeo,
        "Target": targ_yeo,
        "Direction": sub[dir_c].astype(str).values,
    })

    out.to_excel(out_xlsx, index=False)
    print(f"✅ Saved {study_name}:", out_xlsx, "| Rows:", len(out))

# =========================
# Run (compute mapping once, reuse for both studies)
# =========================
ho_label_to_id = load_ho_label_to_id(HO_TXT)
ho_to_yeo = compute_ho_to_yeo7(HO_NII, YEO_NII)

convert_file_to_yeo7(BEDFORD_XLSX, OUT_BEDFORD, "Bedford")
convert_file_to_yeo7(SMIGIELSKI_XLSX, OUT_SMIGIELSKI, "Smigielski")

✅ Saved Bedford: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_yeo_converted/Bedford_2023_Yeo7_converted.xlsx | Rows: 1633
✅ Saved Smigielski: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_yeo_converted/Smigielski_2019_Yeo7_converted.xlsx | Rows: 3


In [7]:
# =========================
# CONVERTING HCP ICA -> Yeo-7 
# =========================
import os, re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

DAI_XLSX = os.path.join(BASE_DIR, "Dai_2023_significant_edges_FINAL.xlsx")

# HCP ICA atlas files (same prefix as .txt)
HCPICA_PREFIX = "conn_networks_hcpica_32"
HCPICA_NII = None
for ext in (".nii.gz", ".nii"):
    p = os.path.join(ATLAS_DIR, HCPICA_PREFIX + ext)
    if os.path.exists(p):
        HCPICA_NII = p
        break
if HCPICA_NII is None:
    raise FileNotFoundError(f"Could not find {HCPICA_PREFIX}.nii(.gz) in: {ATLAS_DIR}")

HCPICA_TXT = os.path.join(ATLAS_DIR, "conn_networks_hcpica_32.txt")
if not os.path.exists(HCPICA_TXT):
    raise FileNotFoundError(f"Could not find {HCPICA_PREFIX}.txt in: {ATLAS_DIR}")

YEO_NII = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR  = os.path.join(BASE_DIR, "data_yeo_converted")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Dai_2023_Yeo7_converted.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a); nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def dai_to_hcpica_base(s: str) -> str:
    """
    Dai-style label -> canonical HCPICA base label with correct casing, e.g.
      'Visual Medial' -> 'Visual.Medial'
      'Default Mode PCC' -> 'DefaultMode.PCC'
      'Language pSTG' -> 'Language.pSTG'
    """
    s0 = clean_ws(s)
    key = s0.lower()

    # normalize network prefix (lowercase)
    key = key.replace("default mode", "defaultmode")
    key = key.replace("dorsal attention", "dorsalattention")
    key = key.replace("frontoparietal", "frontoparietal")
    key = key.replace("sensorimotor", "sensorimotor")
    key = key.replace("salience", "salience")
    key = key.replace("language", "language")
    key = key.replace("visual", "visual")
    key = key.replace("cerebellar", "cerebellar")

    net_case = {
        "defaultmode": "DefaultMode",
        "dorsalattention": "DorsalAttention",
        "frontoparietal": "FrontoParietal",
        "sensorimotor": "SensoriMotor",
        "salience": "Salience",
        "language": "Language",
        "visual": "Visual",
        "cerebellar": "Cerebellar",
    }

    parts = key.split(" ", 1)
    if len(parts) != 2 or parts[0] not in net_case:
        raise ValueError(f"Unrecognized Dai HCP-ICA label: {s!r}")

    net = net_case[parts[0]]
    node_raw = parts[1].strip()  # still lowercase here

    node_case = {
        # Visual / Sensorimotor
        "medial": "Medial",
        "lateral": "Lateral",
        "occipital": "Occipital",
        "superior": "Superior",

        # Default Mode
        "mpfc": "MPFC",
        "pcc": "PCC",
        "lp": "LP",

        # Dorsal Attention
        "fef": "FEF",
        "ips": "IPS",

        # Language
        "ifg": "IFG",
        "pstg": "pSTG",

        # Salience
        "ainsula": "AInsula",
        "rpfc": "RPFC",
        "smg": "SMG",

        # Frontoparietal
        "lpfc": "LPFC",
        "ppc": "PPC",

        # Cerebellar
        "anterior": "Anterior",
        "posterior": "Posterior",
    }

    node_key = node_raw.replace(" ", "")
    node = node_case.get(node_key) or node_case.get(node_raw)
    if node is None:
        raise ValueError(f"Unrecognized Dai HCP-ICA node/subregion: {s!r} -> {node_key!r}")

    return f"{net}.{node}"

def load_hcpica_txt_labels(txt_path: str):
    """
    Parse conn_networks_hcpica_32.txt lines and return:
      base_label -> list of component indices (0-based)
    Strips:
      - trailing coordinate tuple (x,y,z)
      - hemisphere tag "(L)" / "(R)"
    """
    base_to_idxs = {}
    names = []
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                names.append(ln)

    for i, name in enumerate(names):
        # Strip trailing "(x,y,z)"
        name_no_xyz = re.sub(r"\s*\(\s*-?\d+\s*,\s*-?\d+\s*,\s*-?\d+\s*\)\s*$", "", name).strip()
        # Strip hemisphere suffix "(L)" or "(R)"
        base = re.sub(r"\s*\([LR]\)\s*$", "", name_no_xyz).strip()
        base_to_idxs.setdefault(base, []).append(i)

    return base_to_idxs

def compute_hcpica_to_yeo7(hcpica_img, yeo_img, base_to_idxs, n_comps_expected=32):
    """
    For each HCP-ICA base label (e.g., Visual.Lateral), union its L/R components (if present)
    and assign the Yeo-7 label with max Dice overlap.
    """
    hcp = hcpica_img.get_fdata()
    if hcp.ndim != 4:
        raise ValueError(f"Expected HCPICA NIfTI to be 4D. Got shape: {hcp.shape}")
    if n_comps_expected is not None and hcp.shape[-1] < n_comps_expected:
        raise ValueError(f"Expected >= {n_comps_expected} comps. Got {hcp.shape[-1]} comps.")

    # Resample Yeo -> HCPICA grid
    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)
    yeo_rs = resample_to_img(yeo_img, hcpica_img, interpolation="nearest")

    yeo = yeo_rs.get_fdata().astype(int)

    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}
    yeo_masks = {yid: (yeo == yid) for yid in yeo_ids}

    # Component mask: any nonzero voxel belongs to component
    def comp_mask(k):
        return np.abs(hcp[..., k]) > 0

    base_to_yeo = {}
    for base, idxs in base_to_idxs.items():
        hm = np.zeros(yeo.shape, dtype=bool)
        for k in idxs:
            hm |= comp_mask(k)

        best_name, best_d, best_inter = None, -1.0, -1
        for yid, ym in yeo_masks.items():
            inter = np.count_nonzero(hm & ym)
            d = dice(hm, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)
        base_to_yeo[base] = best_name

    return base_to_yeo

def require_map(base, base_to_yeo, row_idx, colname):
    """
    Exact match first; if not found, tries case-insensitive match before failing.
    """
    if base not in base_to_yeo:
        alt = next((k for k in base_to_yeo.keys() if k.lower() == base.lower()), None)
        if alt is not None:
            base = alt
        else:
            known = sorted(base_to_yeo.keys())
            hints = [k for k in known if base.split(".")[0].lower() in k.lower()][:10]
            hint_txt = "\nClosest (same network):\n  - " + "\n  - ".join(hints) if hints else ""
            raise ValueError(
                f"[Mapping error] Row {row_idx}: '{colname}' base label not found.\n"
                f"  Canonical: {base!r}{hint_txt}"
            )

    yeo = base_to_yeo.get(base)
    if yeo not in YEO7:
        raise ValueError(
            f"[Mapping error] Row {row_idx}: '{colname}' got invalid Yeo-7 value: {yeo!r}\n"
            f"  Canonical: {base!r}"
        )
    return yeo

# =========================
# Build HCPICA -> Yeo-7 mapping (Dice)
# =========================
hcpica_img = nib.load(HCPICA_NII)
yeo_img    = nib.load(YEO_NII)

base_to_idxs = load_hcpica_txt_labels(HCPICA_TXT)
base_to_yeo  = compute_hcpica_to_yeo7(hcpica_img, yeo_img, base_to_idxs, n_comps_expected=32)

# =========================
# Convert Dai edges -> Yeo-7 (stop on first failure)
# =========================
df = pd.read_excel(DAI_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

seed_c, targ_c, dir_c, net_c, con_c = "Seed", "Target", "Direction", "Network", "Contrast"

# robust filter: "HCP ICA", "HCPICA", etc.
net_norm = (
    df[net_c].astype(str).str.strip().str.lower()
      .str.replace(r"[^a-z0-9]+", "", regex=True)
)
sub = df[net_norm.eq("hcpica")].copy()

seed_yeo, targ_yeo = [], []
for i, r in sub.iterrows():
    s_base = dai_to_hcpica_base(r[seed_c])   # e.g., Visual.Medial
    t_base = dai_to_hcpica_base(r[targ_c])   # e.g., Language.pSTG
    seed_yeo.append(require_map(s_base, base_to_yeo, i, "Seed"))
    targ_yeo.append(require_map(t_base, base_to_yeo, i, "Target"))

out = pd.DataFrame({
    "Study": "Dai",
    "Contrast": sub[con_c].astype(str).values,
    "Seed": seed_yeo,
    "Target": targ_yeo,
    "Direction": sub[dir_c].astype(str).values,
})

out.to_excel(OUT_XLSX, index=False)
print("✅ Saved:", OUT_XLSX)
print("Rows:", len(out))
print("HCPICA NIfTI used:", HCPICA_NII)

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_yeo_converted/Dai_2023_Yeo7_converted.xlsx
Rows: 31
HCPICA NIfTI used: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/Atlases/conn_networks_hcpica_32.nii


In [8]:
# =========================
# CONVERTING BRODMANN  -> Yeo-7 
# =========================
import os, re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

DEARAUJO_XLSX = os.path.join(BASE_DIR, "DeAraujo_2011_significant_edges_FINAL.xlsx")
BROD_NII      = os.path.join(ATLAS_DIR, "brodmann.nii")
YEO_NII       = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR  = os.path.join(BASE_DIR, "data_yeo_converted")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "DeAraujo_2011_Yeo7_converted.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a); nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def parse_ba(x):
    """
    Accepts: 'BA10', 'ba 10', 'BA 10', etc.
    Returns integer area id (10).
    """
    s = clean_ws(x).lower()
    m = re.match(r"^ba\s*0*(\d+)$", s)
    if not m:
        raise ValueError(f"Unrecognized Brodmann label: {x!r}")
    return int(m.group(1))

def compute_ba_to_yeo7(brod_img, yeo_img):
    """
    For each BA id in brodmann.nii (nonzero integer labels),
    assign Yeo-7 label with max Dice overlap.
    """
    brod = brod_img.get_fdata().astype(int)
    if brod.ndim != 3:
        raise ValueError(f"Expected brodmann atlas to be 3D. Got shape: {brod.shape}")

    # Resample Yeo -> Brodmann grid
    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)
    yeo_rs = resample_to_img(yeo_img, brod_img, interpolation="nearest")

    yeo = yeo_rs.get_fdata().astype(int)

    ba_ids  = [i for i in np.unique(brod) if i != 0]
    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    def best_yeo_for_ba(bid):
        bm = (brod == bid)
        best_name, best_d, best_inter = None, -1.0, -1
        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(bm & ym)
            d = dice(bm, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)
        return best_name

    ba_to_yeo = {bid: best_yeo_for_ba(bid) for bid in ba_ids}
    return ba_to_yeo

def require_map_ba(label, ba_to_yeo, row_idx, colname):
    raw = label
    bid = parse_ba(raw)

    if bid not in ba_to_yeo:
        known = sorted(ba_to_yeo.keys())
        raise ValueError(
            f"[Mapping error] Row {row_idx}: '{colname}' BA id {bid} not found in brodmann.nii.\n"
            f"  Raw value: {raw!r}\n  Known BA ids (sample): {known[:25]}{' ...' if len(known)>25 else ''}"
        )

    yeo = ba_to_yeo.get(bid)
    if yeo not in YEO7:
        raise ValueError(
            f"[Mapping error] Row {row_idx}: '{colname}' BA{bid} mapped to invalid Yeo-7 value: {yeo!r}\n"
            f"  Raw value: {raw!r}"
        )
    return yeo

# =========================
# Build BA -> Yeo-7 mapping (Dice)
# =========================
brod_img = nib.load(BROD_NII)
yeo_img  = nib.load(YEO_NII)
ba_to_yeo = compute_ba_to_yeo7(brod_img, yeo_img)

# =========================
# Convert de Araujo edges -> Yeo-7 (stop on first failure)
# =========================
df = pd.read_excel(DEARAUJO_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

seed_c, targ_c, dir_c, net_c, con_c = "Seed", "Target", "Direction", "Network", "Contrast"

# robust filter: "Brodmann Area", "Brodmann", "BA", etc.
net_norm = (
    df[net_c].astype(str).str.strip().str.lower()
      .str.replace(r"[^a-z0-9]+", "", regex=True)
)
sub = df[net_norm.isin(["brodmannarea", "brodmann", "ba"])].copy()

seed_yeo, targ_yeo = [], []
for i, r in sub.iterrows():
    seed_yeo.append(require_map_ba(r[seed_c], ba_to_yeo, i, "Seed"))
    targ_yeo.append(require_map_ba(r[targ_c], ba_to_yeo, i, "Target"))

out = pd.DataFrame({
    "Study": "de Araujo",
    "Contrast": sub[con_c].astype(str).values,
    "Seed": seed_yeo,
    "Target": targ_yeo,
    "Direction": sub[dir_c].astype(str).values,
})

out.to_excel(OUT_XLSX, index=False)
print("✅ Saved:", OUT_XLSX)
print("Rows:", len(out))
print("Brodmann atlas used:", BROD_NII)

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_yeo_converted/DeAraujo_2011_Yeo7_converted.xlsx
Rows: 29
Brodmann atlas used: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/Atlases/brodmann.nii


In [17]:
# =========================
# CONVERTING Grimm 2018 (AAL seed + MNI target spheres) -> Yeo-7
#   Seed: "R Amygdala" from AAL3v1_1mm.nii.gz
#   Targets: MNI coordinates with radius = Size(target) mm (e.g., 10mm)
# =========================
import os, re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

GRIMM_XLSX = os.path.join(BASE_DIR, "Grimm_2018_significant edges_FINAL.xlsx")

AAL_NII = os.path.join(ATLAS_DIR, "AAL3v1_1mm.nii.gz")      # seed atlas
AAL_TXT = os.path.join(ATLAS_DIR, "AAL3v1_1mm.nii.txt")         # label list (must exist)
YEO_NII = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR  = os.path.join(BASE_DIR, "data_yeo_converted")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Grimm_2018_Yeo7_converted.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a); nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def sphere_mask_on_grid(ref_img: nib.Nifti1Image, center_xyz_mm, radius_mm=10.0):
    """Boolean sphere mask in voxel grid of ref_img."""
    shape = ref_img.shape[:3]
    aff = ref_img.affine
    inv = np.linalg.inv(aff)

    cx, cy, cz = nib.affines.apply_affine(inv, np.array(center_xyz_mm, float))
    cx, cy, cz = float(cx), float(cy), float(cz)

    vx = float(np.sqrt((aff[:3, 0] ** 2).sum()))
    vy = float(np.sqrt((aff[:3, 1] ** 2).sum()))
    vz = float(np.sqrt((aff[:3, 2] ** 2).sum()))

    rx = int(np.ceil(radius_mm / vx))
    ry = int(np.ceil(radius_mm / vy))
    rz = int(np.ceil(radius_mm / vz))

    x0 = max(0, int(np.floor(cx)) - rx); x1 = min(shape[0], int(np.floor(cx)) + rx + 1)
    y0 = max(0, int(np.floor(cy)) - ry); y1 = min(shape[1], int(np.floor(cy)) + ry + 1)
    z0 = max(0, int(np.floor(cz)) - rz); z1 = min(shape[2], int(np.floor(cz)) + rz + 1)

    xs = np.arange(x0, x1)
    ys = np.arange(y0, y1)
    zs = np.arange(z0, z1)
    X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")

    dx = (X - cx) * vx
    dy = (Y - cy) * vy
    dz = (Z - cz) * vz
    local = (dx*dx + dy*dy + dz*dz) <= (radius_mm ** 2)

    m = np.zeros(shape, dtype=bool)
    m[x0:x1, y0:y1, z0:z1] = local
    return m

def yeo_masks_resampled_to(ref_img: nib.Nifti1Image, yeo_path: str):
    """Resample Yeo-7 to ref_img grid and return dict: yeo_name -> bool mask."""
    yeo_img = nib.load(yeo_path)
    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, ref_img, interpolation="nearest")
    yeo = yeo_rs.get_fdata().astype(int)

    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    masks = {}
    for yid in yeo_ids:
        nm = yeo_id_to_name.get(yid)
        if nm:
            masks[nm] = (yeo == yid)
    return masks

def best_yeo_for_mask(mask: np.ndarray, yeo_masks: dict):
    """Return Yeo-7 name with max Dice overlap (tie-break by intersection)."""
    best_name, best_d, best_inter = None, -1.0, -1
    for nm, ym in yeo_masks.items():
        inter = np.count_nonzero(mask & ym)
        d = dice(mask, ym)
        if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
            best_d, best_inter = float(d), int(inter)
            best_name = nm
    return best_name

def load_aal_label_to_id(aal_txt_path: str):
    """
    Expect one label per line; return dict label->id (1..N)
    If your AAL txt includes '1  Precentral_L', tweak parsing accordingly.
    """
    labels = []
    with open(aal_txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            # allow either "ID LABEL" or just "LABEL"
            m = re.match(r"^(\d+)\s+(.+)$", ln)
            if m:
                labels.append(clean_ws(m.group(2)))
            else:
                labels.append(clean_ws(ln))
    return {lab: i+1 for i, lab in enumerate(labels)}

def require_aal_mask(aal_img, aal_label_to_id, seed_label_raw, row_idx):
    """
    Convert "R Amygdala" -> AAL id -> boolean mask.
    IMPORTANT: you may need to adapt SEED_ALIASES to your exact AAL label strings.
    """
    # Map Grimm seed shorthand -> exact AAL label
    SEED_ALIASES = {
        "r amygdala": [
            "Amygdala_R", "Amygdala_R (Amygdala Right)", "Amygdala R", "Right Amygdala",
            "Amygdala_R (Amygdala Right Hemisphere)"
        ]
    }

    key = clean_ws(seed_label_raw).lower()
    candidates = SEED_ALIASES.get(key, [seed_label_raw])

    found = None
    for cand in candidates:
        cand_clean = clean_ws(cand)
        if cand_clean in aal_label_to_id:
            found = cand_clean
            break

    if found is None:
        # helpful hints: substring search for "amygdala" and "r"
        hints = [k for k in aal_label_to_id.keys()
                 if ("amygdala" in k.lower()) or ("amyg" in k.lower())][:20]
        raise ValueError(
            f"[Seed mapping error] Row {row_idx}: seed {seed_label_raw!r} not found in AAL labels.\n"
            f"Try updating SEED_ALIASES with the exact AAL label string.\n"
            f"Example AAL 'amygdala' labels:\n  - " + "\n  - ".join(hints)
        )

    sid = aal_label_to_id[found]
    aal = aal_img.get_fdata().astype(int)
    return (aal == sid)

# =========================
# Run
# =========================
# Load AAL atlas (seed space)
aal_img = nib.load(AAL_NII)
aal_label_to_id = load_aal_label_to_id(AAL_TXT)

# Precompute Yeo masks on AAL grid (we'll use same grid for both seed and targets by
# building target spheres directly on AAL grid too)
yeo_masks = yeo_masks_resampled_to(aal_img, YEO_NII)

df = pd.read_excel(GRIMM_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

# Expected columns in your Grimm sheet
study_c = "Study"
con_c   = "Contrast"
seed_c  = "Seed"
targ_c  = "Target"
x_c, y_c, z_c = "x (target)", "y (target)", "z (target)"
rad_c   = "Size (target)"
dir_c   = "Direction"

seed_out, targ_out = [], []

for i, r in df.iterrows():
    # ----- Seed (AAL ROI) -> Yeo-7
    seed_mask = require_aal_mask(aal_img, aal_label_to_id, r[seed_c], row_idx=i)
    seed_yeo = best_yeo_for_mask(seed_mask, yeo_masks)
    if seed_yeo not in YEO7:
        raise ValueError(f"[Seed Yeo mapping error] Row {i}: seed {r[seed_c]!r} mapped to {seed_yeo!r}")

    # ----- Target (sphere around coordinate) -> Yeo-7
    x, y, z = float(r[x_c]), float(r[y_c]), float(r[z_c])
    rad = float(r[rad_c])
    targ_mask = sphere_mask_on_grid(aal_img, (x, y, z), radius_mm=rad)
    targ_yeo = best_yeo_for_mask(targ_mask, yeo_masks)
    if targ_yeo not in YEO7:
        raise ValueError(f"[Target Yeo mapping error] Row {i}: target coord {(x,y,z)} mapped to {targ_yeo!r}")

    seed_out.append(seed_yeo)
    targ_out.append(targ_yeo)

out = pd.DataFrame({
    "Study": df[study_c].astype(str).values,
    "Contrast": df[con_c].astype(str).values,
    "Seed": seed_out,
    "Target": targ_out,
    "Direction": df[dir_c].astype(str).values,
})

out.to_excel(OUT_XLSX, index=False)
print("✅ Saved:", OUT_XLSX)
print("Rows:", len(out))
print("Seed atlas used:", AAL_NII)
print("Yeo atlas used:", YEO_NII)

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_yeo_converted/Grimm_2018_Yeo7_converted.xlsx
Rows: 2
Seed atlas used: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/Atlases/AAL3v1_1mm.nii.gz
Yeo atlas used: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/Atlases/Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii


In [24]:
# =========================
# CONVERTING Madsen 2021 (Raichle ROI networks) -> Yeo-7
# =========================
import os
import numpy as np
import pandas as pd
import nibabel as nib

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

COORD_XLSX = os.path.join(BASE_DIR, "Madsen_2021_Network_Coordinates.xlsx")
EDGES_XLSX = os.path.join(BASE_DIR, "Madsen_2021_significant_edges_FINAL.xlsx")

YEO_NII = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR  = os.path.join(BASE_DIR, "data_yeo_converted")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Madsen_2021_Yeo7_converted.xlsx")

RADIUS_MM = 10

YEO7 = [
    "Visual",
    "Somatomotor",
    "DorsalAttention",
    "VentralAttention",
    "Limbic",
    "Frontoparietal",
    "Default",
]

# =========================
# Dice
# =========================
def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0 if na + nb == 0 else (2 * inter) / (na + nb)

# =========================
# Sphere generator
# =========================
def sphere_mask(ref_img, coord, radius):

    shape = ref_img.shape[:3]
    affine = ref_img.affine
    inv = np.linalg.inv(affine)

    cx, cy, cz = nib.affines.apply_affine(inv, coord)

    vx = np.linalg.norm(affine[:3,0])
    vy = np.linalg.norm(affine[:3,1])
    vz = np.linalg.norm(affine[:3,2])

    rx = int(np.ceil(radius/vx))
    ry = int(np.ceil(radius/vy))
    rz = int(np.ceil(radius/vz))

    mask = np.zeros(shape,dtype=bool)

    for x in range(int(cx-rx),int(cx+rx)):
        for y in range(int(cy-ry),int(cy+ry)):
            for z in range(int(cz-rz),int(cz+rz)):

                if x<0 or y<0 or z<0: continue
                if x>=shape[0] or y>=shape[1] or z>=shape[2]: continue

                dx=(x-cx)*vx
                dy=(y-cy)*vy
                dz=(z-cz)*vz

                if dx*dx+dy*dy+dz*dz <= radius**2:
                    mask[x,y,z]=True

    return mask

# =========================
# Load coordinate table
# =========================
coords = pd.read_excel(COORD_XLSX)

coords["network_norm"] = (
    coords["Network"]
    .str.lower()
    .str.replace(" ","")
)

# =========================
# Load Yeo atlas
# =========================
yeo_img = nib.load(YEO_NII)
yeo = yeo_img.get_fdata().astype(int)

yeo_ids = sorted([i for i in np.unique(yeo) if i!=0])[:7]

yeo_masks = {}
for i,yid in enumerate(yeo_ids):
    yeo_masks[YEO7[i]] = yeo==yid

# =========================
# Build Raichle network masks
# =========================
raichle_masks = {}

for network in coords["Network"].unique():

    sub = coords[coords["Network"]==network]

    mask = np.zeros(yeo_img.shape[:3],dtype=bool)

    for _,r in sub.iterrows():

        coord = (r["X"],r["Y"],r["Z"])

        mask |= sphere_mask(yeo_img,coord,RADIUS_MM)

    raichle_masks[network]=mask

# =========================
# Map Raichle networks -> Yeo
# =========================
raichle_to_yeo={}

for net,mask in raichle_masks.items():

    best=None
    best_d=-1

    for yn,ymask in yeo_masks.items():

        d=dice(mask,ymask)

        if d>best_d:
            best_d=d
            best=yn

    raichle_to_yeo[net]=best

print("Raichle -> Yeo mapping")
print(raichle_to_yeo)

# =========================
# Network abbreviations in Madsen edges
# =========================
ABBREV_MAP={
"DMN":"Default mode network",
"DAN":"Dorsal attention network",
"ECN":"Executive control network",
"SAN":"Salience network",
"SMN":"Sensorimotor network",
"VN":"Visual network",
"AN":"Auditory network"
}

# =========================
# Convert edges
# =========================
edges=pd.read_excel(EDGES_XLSX)

seed_out=[]
targ_out=[]

for _,r in edges.iterrows():

    seed_raichle=ABBREV_MAP[r["Seed"]]
    targ_raichle=ABBREV_MAP[r["Target"]]

    seed_out.append(raichle_to_yeo[seed_raichle])
    targ_out.append(raichle_to_yeo[targ_raichle])

out=pd.DataFrame({
"Study":"Madsen",
"Contrast":edges["Contrast"],
"Seed":seed_out,
"Target":targ_out,
"Direction":edges["Direction"]
})

out.to_excel(OUT_XLSX,index=False)

print("Saved:",OUT_XLSX)

Raichle -> Yeo mapping
{'Default mode network': 'Somatomotor', 'Dorsal attention network': 'DorsalAttention', 'Executive control network': 'VentralAttention', 'Salience network': 'Somatomotor', 'Sensorimotor network': 'DorsalAttention', 'Visual network': 'Somatomotor', 'Auditory network': 'Limbic'}
Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_yeo_converted/Madsen_2021_Yeo7_converted.xlsx


In [27]:
# =========================
# CONVERTING Smith 2009 (Roseman 2014 + Mason 2020) -> Yeo-7
#   FIXED: threshold Smith component maps before Dice
# =========================
import os, re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

ROSEMAN_XLSX = os.path.join(BASE_DIR, "Roseman_2014_significant_edges_FINAL.xlsx")
MASON_XLSX   = os.path.join(BASE_DIR, "Mason_2020_significant_edges_FINAL.xlsx")

SMITH_NII    = os.path.join(ATLAS_DIR, "PNAS_Smith09_rsn10.nii.gz")
SMITH_LABELS = os.path.join(ATLAS_DIR, "SMITH09_RSN10_labels.txt")
YEO_NII      = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR = os.path.join(BASE_DIR, "data_yeo_converted")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_ROSEMAN = os.path.join(OUT_DIR, "Roseman_2014_Yeo7_converted.xlsx")
OUT_MASON   = os.path.join(OUT_DIR, "Mason_2020_Yeo7_converted.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# IMPORTANT: threshold Smith ICA maps before making masks
SMITH_Z_THRESHOLD = 2.3

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def norm(s):
    s = clean_ws(s).lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

# =========================
# Load Smith RSN10 canonical labels
# =========================
def load_smith_labels(labels_path):
    id_to_label = {}
    with open(labels_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            m = re.match(r"^(\d+)\s+(.+)$", ln)
            if not m:
                continue
            rid = int(m.group(1))
            lab = clean_ws(m.group(2))
            id_to_label[rid] = lab
    label_to_id = {norm(v): k for k, v in id_to_label.items()}
    return id_to_label, label_to_id

# =========================
# Label normalization for Mason + Roseman
# =========================
SMITH_ALIAS = {
    # canonical/common
    "cerebellum": "Cerebellum",
    "auditory": "Auditory",
    "aud": "Auditory",
    "sensorimotor": "Sensorimotor",
    "sm": "Sensorimotor",
    "dmn": "Default mode network",
    "defaultmodenetwork": "Default mode network",
    "executive": "Executive control",
    "executivecontrol": "Executive control",
    "ecn": "Executive control",
    "lfp": "Left frontoparietal",
    "leftfrontoparietal": "Left frontoparietal",
    "frontoparietal1": "Left frontoparietal",
    "rfp": "Right frontoparietal",
    "rightfrontoparietal": "Right frontoparietal",
    "frontoparietal2": "Right frontoparietal",

    # Mason visuals
    "visual1": "Visual medial",
    "visual2": "Visual occipital pole",
    "visual3": "Visual lateral",

    # Roseman visuals
    "vism": "Visual medial",
    "viso": "Visual occipital pole",
    "visl": "Visual lateral",

    # Roseman extras collapsed to Smith RSN10
    "dan": "Executive control",
    "dan2": "Executive control",
    "dmn2": "Default mode network",
}

def canonical_smith_label(x, row_idx, colname, study_name, label_to_id):
    raw = clean_ws(x)
    k = norm(raw)

    if k in label_to_id:
        return next(v for kk, v in [(norm(v), v) for v in label_to_id.keys()] if False)

    if k in SMITH_ALIAS:
        canon = SMITH_ALIAS[k]
        if norm(canon) in label_to_id:
            return canon

    # direct canonical match fallback
    for rid, lab in id_to_label.items():
        if norm(lab) == k:
            return lab

    hints = [h for h in sorted(set(list(label_to_id.keys()) + list(SMITH_ALIAS.keys()))) if k in h or h in k][:10]
    hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints) if hints else ""
    raise ValueError(
        f"[{study_name}] Row {row_idx}: could not map '{colname}' label to Smith RSN10.\n"
        f"  Raw value: {raw!r}\n"
        f"  Normalized: {k!r}{hint_txt}"
    )

# =========================
# Smith RSN10 -> Yeo-7 via Dice
# =========================
def compute_smith_to_yeo7(smith_nii_path, yeo_nii_path, zthr=2.3):
    smith_img = nib.load(smith_nii_path)
    yeo_img   = nib.load(yeo_nii_path)

    smith = smith_img.get_fdata()
    if smith.ndim != 4:
        raise ValueError(f"Expected Smith atlas to be 4D. Got shape: {smith.shape}")
    if smith.shape[-1] < 10:
        raise ValueError(f"Expected >=10 Smith components. Got: {smith.shape[-1]}")

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, smith_img, interpolation="nearest")
    yeo = yeo_rs.get_fdata().astype(int)

    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    smith_to_yeo = {}
    debug_rows = []

    for k in range(10):
        comp = smith[..., k]

        # FIX: threshold the Smith spatial map
        comp_mask = comp > zthr

        if np.count_nonzero(comp_mask) == 0:
            raise ValueError(
                f"Smith component {k+1} has zero voxels above threshold {zthr}. "
                f"Try lowering SMITH_Z_THRESHOLD (e.g., 2.0 or 1.5)."
            )

        best_name, best_d, best_inter = None, -1.0, -1

        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(comp_mask & ym)
            d = dice(comp_mask, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)

        smith_to_yeo[k + 1] = best_name
        debug_rows.append({
            "Smith_Component": k + 1,
            "Best_Yeo7": best_name,
            "Dice": best_d,
            "Intersect_Voxels": best_inter,
            "Component_Voxels": int(np.count_nonzero(comp_mask))
        })

    debug_df = pd.DataFrame(debug_rows)
    print("\nSmith -> Yeo-7 mapping:")
    print(debug_df.to_string(index=False))

    return smith_to_yeo

# =========================
# Convert one study file
# =========================
def convert_smith_study(infile, outfile, study_name, label_to_id, smith_to_yeo, id_to_label):
    df = pd.read_excel(infile)
    df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

    seed_c, targ_c, dir_c = "Seed", "Target", "Direction"
    con_c = "Contrast"
    net_c = "Network"

    if net_c in df.columns:
        net_norm = (
            df[net_c].astype(str).str.strip().str.lower()
            .str.replace(r"[^a-z0-9]+", "", regex=True)
        )
        sub = df[net_norm.isin(["smith2009", "smith", "smithrsn10"])].copy()
        if len(sub) == 0:
            sub = df.copy()
    else:
        sub = df.copy()

    def canonical_label(x, row_idx, colname):
        raw = clean_ws(x)
        k = norm(raw)

        for rid, lab in id_to_label.items():
            if norm(lab) == k:
                return lab

        if k in SMITH_ALIAS:
            canon = SMITH_ALIAS[k]
            for rid, lab in id_to_label.items():
                if norm(lab) == norm(canon):
                    return lab

        hints = [lab for rid, lab in id_to_label.items() if k in norm(lab) or norm(lab) in k][:10]
        hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints) if hints else ""
        raise ValueError(
            f"[{study_name}] Row {row_idx}: could not map '{colname}' label to Smith RSN10.\n"
            f"  Raw value: {raw!r}\n"
            f"  Normalized: {k!r}{hint_txt}"
        )

    seed_out, targ_out = [], []

    for i, r in sub.iterrows():
        seed_lab = canonical_label(r[seed_c], i, "Seed")
        targ_lab = canonical_label(r[targ_c], i, "Target")

        sid = label_to_id[norm(seed_lab)]
        tid = label_to_id[norm(targ_lab)]

        s_yeo = smith_to_yeo.get(sid)
        t_yeo = smith_to_yeo.get(tid)

        if s_yeo not in YEO7:
            raise ValueError(f"[{study_name}] Row {i}: Seed {seed_lab!r} mapped to invalid Yeo-7 value: {s_yeo!r}")
        if t_yeo not in YEO7:
            raise ValueError(f"[{study_name}] Row {i}: Target {targ_lab!r} mapped to invalid Yeo-7 value: {t_yeo!r}")

        seed_out.append(s_yeo)
        targ_out.append(t_yeo)

    out = pd.DataFrame({
        "Study": study_name,
        "Contrast": sub[con_c].astype(str).values,
        "Seed": seed_out,
        "Target": targ_out,
        "Direction": sub[dir_c].astype(str).values,
    })

    out.to_excel(outfile, index=False)
    print(f"\n✅ Saved {study_name}: {outfile} | Rows: {len(out)}")

# =========================
# Run
# =========================
id_to_label, label_to_id = load_smith_labels(SMITH_LABELS)
smith_to_yeo = compute_smith_to_yeo7(SMITH_NII, YEO_NII, zthr=SMITH_Z_THRESHOLD)

convert_smith_study(ROSEMAN_XLSX, OUT_ROSEMAN, "Roseman", label_to_id, smith_to_yeo, id_to_label)
convert_smith_study(MASON_XLSX, OUT_MASON, "Mason", label_to_id, smith_to_yeo, id_to_label)


Smith -> Yeo-7 mapping:
 Smith_Component        Best_Yeo7     Dice  Intersect_Voxels  Component_Voxels
               1           Visual 0.278647              3464             16664
               2           Visual 0.198226              2090             12888
               3           Visual 0.234508              3894             25011
               4          Default 0.342841              6178             19187
               5           Visual 0.062025               706             14566
               6      Somatomotor 0.244239              4086             24959
               7 VentralAttention 0.217414              3326             23180
               8   Frontoparietal 0.146063              3303             34771
               9   Frontoparietal 0.219659              4152             27348
              10   Frontoparietal 0.183292              3278             25312

✅ Saved Roseman: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_

In [29]:
# =========================
# CONVERTING Palhano-Fontes 2015 (MNI seed spheres + MNI target spheres) -> Yeo-7
#   - uses 10 mm spheres around BOTH seed and target coordinates
#   - maps each sphere to Yeo-7 via Dice overlap
#   - stops + flags the first unmappable row
#   - outputs ONE Excel with columns: Study, Contrast, Seed, Target, Direction
# =========================
import os
import numpy as np
import pandas as pd
import nibabel as nib

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

PALHANO_XLSX = os.path.join(BASE_DIR, "PalhanoFontes_2015_significant edges_FINAL.xlsx")
YEO_NII      = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR  = os.path.join(BASE_DIR, "data_yeo_converted")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "PalhanoFontes_2015_Yeo7_converted.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# =========================
# Helpers
# =========================
def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def sphere_mask_on_grid(ref_img: nib.Nifti1Image, center_xyz_mm, radius_mm=10.0):
    """Create a boolean sphere mask in the voxel grid of ref_img."""
    shape = ref_img.shape[:3]
    aff = ref_img.affine
    inv = np.linalg.inv(aff)

    cx, cy, cz = nib.affines.apply_affine(inv, np.array(center_xyz_mm, dtype=float))
    cx, cy, cz = float(cx), float(cy), float(cz)

    vx = float(np.linalg.norm(aff[:3, 0]))
    vy = float(np.linalg.norm(aff[:3, 1]))
    vz = float(np.linalg.norm(aff[:3, 2]))

    rx = int(np.ceil(radius_mm / vx))
    ry = int(np.ceil(radius_mm / vy))
    rz = int(np.ceil(radius_mm / vz))

    x0 = max(0, int(np.floor(cx)) - rx)
    x1 = min(shape[0], int(np.floor(cx)) + rx + 1)
    y0 = max(0, int(np.floor(cy)) - ry)
    y1 = min(shape[1], int(np.floor(cy)) + ry + 1)
    z0 = max(0, int(np.floor(cz)) - rz)
    z1 = min(shape[2], int(np.floor(cz)) + rz + 1)

    xs = np.arange(x0, x1)
    ys = np.arange(y0, y1)
    zs = np.arange(z0, z1)
    X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")

    dx = (X - cx) * vx
    dy = (Y - cy) * vy
    dz = (Z - cz) * vz

    local = (dx*dx + dy*dy + dz*dz) <= (radius_mm ** 2)

    mask = np.zeros(shape, dtype=bool)
    mask[x0:x1, y0:y1, z0:z1] = local
    return mask

def get_yeo_masks(yeo_img):
    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yd = np.squeeze(yd, axis=-1)
        yeo_img = nib.Nifti1Image(yd, affine=yeo_img.affine)

    yeo = yd.astype(int)
    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]

    yeo_masks = {}
    for i, yid in enumerate(yeo_ids):
        yeo_masks[YEO7[i]] = (yeo == yid)

    return yeo_masks

def best_yeo_for_mask(mask, yeo_masks):
    best_name = None
    best_d = -1.0
    best_inter = -1

    for name, ym in yeo_masks.items():
        inter = np.count_nonzero(mask & ym)
        d = dice(mask, ym)
        if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
            best_name = name
            best_d = float(d)
            best_inter = int(inter)

    return best_name

# =========================
# Load data + atlas
# =========================
df = pd.read_excel(PALHANO_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

yeo_img = nib.load(YEO_NII)
yeo_masks = get_yeo_masks(yeo_img)

# Expected columns
seed_x_c = "x (seed)"
seed_y_c = "y (seed)"
seed_z_c = "z (seed)"
seed_r_c = "Size (seed)"

targ_x_c = "x (target)"
targ_y_c = "y (target)"
targ_z_c = "z (target)"
targ_r_c = "Size (target)"

study_c = "Study"
con_c   = "Contrast"
dir_c   = "Direction"

# =========================
# Convert row-by-row
# =========================
seed_out = []
targ_out = []

for i, r in df.iterrows():
    # Seed sphere
    s_xyz = (float(r[seed_x_c]), float(r[seed_y_c]), float(r[seed_z_c]))
    s_rad = float(r[seed_r_c])
    seed_mask = sphere_mask_on_grid(yeo_img, s_xyz, radius_mm=s_rad)
    seed_yeo = best_yeo_for_mask(seed_mask, yeo_masks)

    if seed_yeo not in YEO7:
        raise ValueError(
            f"[Seed mapping error] Row {i}: seed coord {s_xyz} with radius {s_rad} mapped to invalid Yeo label {seed_yeo!r}"
        )

    # Target sphere
    t_xyz = (float(r[targ_x_c]), float(r[targ_y_c]), float(r[targ_z_c]))
    t_rad = float(r[targ_r_c])
    targ_mask = sphere_mask_on_grid(yeo_img, t_xyz, radius_mm=t_rad)
    targ_yeo = best_yeo_for_mask(targ_mask, yeo_masks)

    if targ_yeo not in YEO7:
        raise ValueError(
            f"[Target mapping error] Row {i}: target coord {t_xyz} with radius {t_rad} mapped to invalid Yeo label {targ_yeo!r}"
        )

    seed_out.append(seed_yeo)
    targ_out.append(targ_yeo)

# =========================
# Save converted file
# =========================
out = pd.DataFrame({
    "Study": df[study_c].astype(str).values,
    "Contrast": df[con_c].astype(str).values,
    "Seed": seed_out,
    "Target": targ_out,
    "Direction": df[dir_c].astype(str).values,
})

out.to_excel(OUT_XLSX, index=False)

print("✅ Saved:", OUT_XLSX)
print("Rows:", len(out))
print(out)

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_yeo_converted/PalhanoFontes_2015_Yeo7_converted.xlsx
Rows: 2
            Study              Contrast     Seed   Target Direction
0  Palhano-Fontes  Post > Pre Ayahuasca  Default  Default  Decrease
1  Palhano-Fontes  Post > Pre Ayahuasca  Default  Default  Decrease


In [37]:
# =========================
# CONVERTING Pasquini 2020 (updated network coordinates) -> Yeo-7
#
# Inputs:
#   1) Pasquini_2020_Network_Coordinates.xlsx
#      columns expected: Network, x, y, z, Region
#   2) Pasquini_2020_significant edges_FINAL.xlsx
#      columns expected: Study, Contrast, Seed, Target, Direction
#
# Method:
#   - Build one binary mask per network by UNION of 10mm-radius spheres at its ROI coordinates
#   - Compute Dice overlap between each network mask and each Yeo-7 network mask
#   - Map each Pasquini network -> best Yeo-7 network
#   - Convert Pasquini edges -> Yeo-7 edges
#
# Output:
#   data_yeo_converted/Pasquini_2020_Yeo7_converted.xlsx
#   columns: Study, Contrast, Seed, Target, Direction
# =========================
import os
import re
import numpy as np
import pandas as pd
import nibabel as nib

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

COORD_XLSX = os.path.join(BASE_DIR, "Pasquini_2020_Network_Coordinates.xlsx")
EDGES_XLSX = os.path.join(BASE_DIR, "Pasquini_2020_significant edges_FINAL.xlsx")
YEO_NII    = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR  = os.path.join(BASE_DIR, "data_yeo_converted")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Pasquini_2020_Yeo7_converted.xlsx")

RADIUS_MM = 10.0

YEO7 = [
    "Visual",
    "Somatomotor",
    "DorsalAttention",
    "VentralAttention",
    "Limbic",
    "Frontoparietal",
    "Default",
]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def norm_key(s):
    return re.sub(r"[^a-z0-9]+", "", clean_ws(s).lower())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def sphere_mask_on_grid(ref_img: nib.Nifti1Image, center_xyz_mm, radius_mm=10.0):
    """Boolean sphere mask in voxel grid of ref_img."""
    shape = ref_img.shape[:3]
    aff = ref_img.affine
    inv = np.linalg.inv(aff)

    cx, cy, cz = nib.affines.apply_affine(inv, np.array(center_xyz_mm, dtype=float))
    cx, cy, cz = float(cx), float(cy), float(cz)

    vx = float(np.linalg.norm(aff[:3, 0]))
    vy = float(np.linalg.norm(aff[:3, 1]))
    vz = float(np.linalg.norm(aff[:3, 2]))

    rx = int(np.ceil(radius_mm / vx))
    ry = int(np.ceil(radius_mm / vy))
    rz = int(np.ceil(radius_mm / vz))

    x0 = max(0, int(np.floor(cx)) - rx)
    x1 = min(shape[0], int(np.floor(cx)) + rx + 1)
    y0 = max(0, int(np.floor(cy)) - ry)
    y1 = min(shape[1], int(np.floor(cy)) + ry + 1)
    z0 = max(0, int(np.floor(cz)) - rz)
    z1 = min(shape[2], int(np.floor(cz)) + rz + 1)

    xs = np.arange(x0, x1)
    ys = np.arange(y0, y1)
    zs = np.arange(z0, z1)
    X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")

    dx = (X - cx) * vx
    dy = (Y - cy) * vy
    dz = (Z - cz) * vz
    local = (dx * dx + dy * dy + dz * dz) <= (radius_mm ** 2)

    mask = np.zeros(shape, dtype=bool)
    mask[x0:x1, y0:y1, z0:z1] = local
    return mask

def get_yeo_masks(yeo_img):
    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yd = np.squeeze(yd, axis=-1)

    yeo = yd.astype(int)
    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])

    if yeo_ids != [1, 2, 3, 4, 5, 6, 7]:
        raise ValueError(f"Unexpected Yeo labels: {yeo_ids}")

    return {YEO7[i-1]: (yeo == i) for i in yeo_ids}

def best_yeo_for_mask(mask, yeo_masks):
    best_name = None
    best_d = -1.0
    best_inter = -1
    for nm, ym in yeo_masks.items():
        inter = np.count_nonzero(mask & ym)
        d = dice(mask, ym)
        if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
            best_name = nm
            best_d = float(d)
            best_inter = int(inter)
    return best_name

# =========================
# Load coordinate table
# =========================
coords = pd.read_excel(COORD_XLSX)
coords.columns = coords.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

# normalize x/y/z capitalization if needed
rename_map = {}
for c in coords.columns:
    if norm_key(c) == "x":
        rename_map[c] = "x"
    elif norm_key(c) == "y":
        rename_map[c] = "y"
    elif norm_key(c) == "z":
        rename_map[c] = "z"
    elif norm_key(c) == "network":
        rename_map[c] = "Network"
    elif norm_key(c) == "region":
        rename_map[c] = "Region"
coords = coords.rename(columns=rename_map)

required_coord_cols = {"Network", "x", "y", "z"}
missing = required_coord_cols - set(coords.columns)
if missing:
    raise ValueError(f"Coordinate file missing columns: {missing}. Found: {list(coords.columns)}")

coords["Network"] = coords["Network"].astype(str).str.strip()

# =========================
# Load Yeo atlas + masks
# =========================
yeo_img = nib.load(YEO_NII)
yeo_masks = get_yeo_masks(yeo_img)

# =========================
# Build network masks from 10mm spheres
# =========================
network_masks = {}

for network in coords["Network"].dropna().astype(str).str.strip().unique():
    sub = coords[coords["Network"].astype(str).str.strip() == network]

    mask = np.zeros(yeo_img.shape[:3], dtype=bool)
    for _, r in sub.iterrows():
        xyz = (float(r["x"]), float(r["y"]), float(r["z"]))
        mask |= sphere_mask_on_grid(yeo_img, xyz, radius_mm=RADIUS_MM)

    if np.count_nonzero(mask) == 0:
        raise ValueError(f"Network {network!r} produced an empty sphere mask.")
    network_masks[network] = mask

# =========================
# Map Pasquini networks -> Yeo-7 via Dice
# =========================
network_to_yeo = {}
print("Pasquini network -> Yeo-7 mapping (Dice-based):")
for network, mask in network_masks.items():
    best = best_yeo_for_mask(mask, yeo_masks)
    if best not in YEO7:
        raise ValueError(f"Network {network!r} mapped to invalid Yeo-7 label: {best!r}")
    network_to_yeo[network] = best
    print(f"  {network} -> {best}")

# =========================
# Convert edges
# =========================
edges = pd.read_excel(EDGES_XLSX)
edges.columns = edges.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

required_edge_cols = {"Study", "Contrast", "Seed", "Target", "Direction"}
missing = required_edge_cols - set(edges.columns)
if missing:
    raise ValueError(f"Edge file missing columns: {missing}. Found: {list(edges.columns)}")

def require_network_to_yeo(label, row_idx, colname):
    raw = clean_ws(label)
    if raw in network_to_yeo:
        return network_to_yeo[raw]

    raw_norm = norm_key(raw)
    for k in network_to_yeo:
        if norm_key(k) == raw_norm:
            return network_to_yeo[k]

    hints = [k for k in network_to_yeo if raw_norm in norm_key(k) or norm_key(k) in raw_norm][:10]
    hint_txt = ""
    if hints:
        hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints)

    raise ValueError(
        f"[Mapping error] Row {row_idx}: could not map {colname} label {raw!r} "
        f"to a network in Pasquini_2020_Network_Coordinates.{hint_txt}"
    )

seed_out = []
targ_out = []

for i, r in edges.iterrows():
    seed_out.append(require_network_to_yeo(r["Seed"], i, "Seed"))
    targ_out.append(require_network_to_yeo(r["Target"], i, "Target"))

out = pd.DataFrame({
    "Study": edges["Study"].astype(str).values,
    "Contrast": edges["Contrast"].astype(str).values,
    "Seed": seed_out,
    "Target": targ_out,
    "Direction": edges["Direction"].astype(str).values,
})

out.to_excel(OUT_XLSX, index=False)

print("\n✅ Saved:", OUT_XLSX)
print("Rows:", len(out))
print(out)

Pasquini network -> Yeo-7 mapping (Dice-based):
  Salience -> VentralAttention
  Default Mode -> Default
  Visual -> Visual
  Sensorimotor -> Somatomotor

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_yeo_converted/Pasquini_2020_Yeo7_converted.xlsx
Rows: 3
      Study              Contrast              Seed            Target  \
0  Pasquini  Ayahuasca > Placebo   VentralAttention  VentralAttention   
1  Pasquini  Ayahuasca > Placebo            Default           Default   
2  Pasquini  Ayahuasca > Placebo   VentralAttention           Default   

  Direction  
0  Increase  
1  Decrease  
2  Increase  


In [4]:
# =========================
# CONVERTING Kaelen 2016 (Harvard-Oxford seed + MNI target spheres) -> Yeo-7
#   Seed: Harvard-Oxford label abbreviation(s) from CONN HO atlas
#   Targets: MNI coordinates -> 10 mm spheres
# =========================
import os, re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

KAELEN_XLSX = os.path.join(BASE_DIR, "Kaelen_2016_significant edges_FINAL.xlsx")

HO_NII  = os.path.join(ATLAS_DIR, "conn_atlas_harvardoxford.nii")
HO_TXT  = os.path.join(ATLAS_DIR, "conn_atlas_harvardoxford.txt")
YEO_NII = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR  = os.path.join(BASE_DIR, "data_yeo_converted")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Kaelen_2016_Yeo7_converted.xlsx")

YEO7 = ["Visual", "Somatomotor", "DorsalAttention", "VentralAttention", "Limbic", "Frontoparietal", "Default"]
TARGET_RADIUS_MM = 10.0

# In your CONN Harvard-Oxford text file, parahippocampal cortex is represented as
#   aPaHC r / aPaHC l / pPaHC r / pPaHC l
# Kaelen's seed label is PHC, so by default this script treats PHC as bilateral
# posterior parahippocampal cortex.
PHC_HO_LABELS = ["pPaHC l", "pPaHC r"]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def norm(s):
    return re.sub(r"[^a-z0-9]+", "", clean_ws(s).lower())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def load_ho_label_to_id(txt_path):
    labels = []
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                # Keep only the short CONN label before the parenthetical long name.
                short_lab = clean_ws(ln.split("(", 1)[0])
                labels.append(short_lab)
    return {lab: i + 1 for i, lab in enumerate(labels)}

def compute_ho_to_yeo7(ho_nii_path, yeo_nii_path):
    ho_img = nib.load(ho_nii_path)
    yeo_img = nib.load(yeo_nii_path)

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, ho_img, interpolation="nearest", force_resample=True, copy_header=True)
    ho = ho_img.get_fdata().astype(int)
    yeo = yeo_rs.get_fdata().astype(int)

    ho_ids = [i for i in np.unique(ho) if i != 0]
    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    ho_to_yeo = {}
    debug_rows = []

    for hid in ho_ids:
        hm = (ho == hid)
        best_name, best_d, best_inter = None, -1.0, -1

        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(hm & ym)
            d = dice(hm, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)

        ho_to_yeo[hid] = best_name
        debug_rows.append({
            "HO_ID": hid,
            "Best_Yeo7": best_name,
            "Dice": best_d,
            "Intersect_Voxels": best_inter,
            "HO_Voxels": int(np.count_nonzero(hm)),
        })

    return ho_to_yeo, pd.DataFrame(debug_rows), ho_img

def get_yeo_masks(ref_img, yeo_path):
    yeo_img = nib.load(yeo_path)
    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, ref_img, interpolation="nearest", force_resample=True, copy_header=True)
    yeo = yeo_rs.get_fdata().astype(int)

    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_masks = {}
    for i, yid in enumerate(yeo_ids):
        yeo_masks[YEO7[i]] = (yeo == yid)
    return yeo_masks

def best_yeo_for_mask(mask, yeo_masks, row_idx=None, colname=None):
    if np.count_nonzero(mask) == 0:
        where = f" row {row_idx}" if row_idx is not None else ""
        what = f" ({colname})" if colname is not None else ""
        raise ValueError(f"Got an empty mask for{where}{what}.")

    best_name, best_d, best_inter = None, -1.0, -1
    for yname, ym in yeo_masks.items():
        inter = np.count_nonzero(mask & ym)
        d = dice(mask, ym)
        if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
            best_d, best_inter = float(d), int(inter)
            best_name = yname
    return best_name, best_d, best_inter

def sphere_mask_on_grid(ref_img: nib.Nifti1Image, center_xyz_mm, radius_mm=10.0):
    shape = ref_img.shape[:3]
    aff = ref_img.affine

    ii, jj, kk = np.indices(shape)
    vox = np.column_stack([ii.ravel(), jj.ravel(), kk.ravel()])
    xyz = nib.affines.apply_affine(aff, vox)
    dist = np.sqrt(((xyz - np.array(center_xyz_mm, dtype=float)) ** 2).sum(axis=1))
    return (dist <= float(radius_mm)).reshape(shape)

# =========================
# Kaelen-specific label handling
# =========================
KAELEN_SEED_ALIAS = {
    "phc": PHC_HO_LABELS,
    "parahippocampal": PHC_HO_LABELS,
    "parahippocampalcortex": PHC_HO_LABELS,
    "parahippocampalgyrus": PHC_HO_LABELS,
    "ppahc": PHC_HO_LABELS,
}

def resolve_ho_labels(raw_label, ho_label_to_id, alias_map=None):
    raw = clean_ws(raw_label)
    k = norm(raw)

    if alias_map and k in alias_map:
        labs = alias_map[k]
    else:
        labs = [raw]

    if isinstance(labs, str):
        labs = [labs]

    resolved = []
    for lab in labs:
        if lab in ho_label_to_id:
            resolved.append(lab)
            continue

        nk = norm(lab)
        exact_norm = [h for h in ho_label_to_id if norm(h) == nk]
        if exact_norm:
            resolved.append(exact_norm[0])
            continue

        if "pahc" in nk or "parahippocampal" in nk:
            cands = [h for h in ho_label_to_id if ("pahc" in norm(h) or "parahippocampal" in norm(h))]
            cands = sorted(cands, key=lambda x: (("ppahc" not in norm(x) and "posterior" not in norm(x)), x))
            if cands:
                resolved.extend(cands[:2] if len(cands) >= 2 else cands)
                continue

        hints = [h for h in ho_label_to_id if nk in norm(h) or norm(h) in nk][:10]
        hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints) if hints else ""
        raise ValueError(
            f"[Kaelen] Could not resolve HO label.\n"
            f"  Raw value: {raw!r}\n"
            f"  Candidate: {lab!r}{hint_txt}"
        )

    out = []
    seen = set()
    for x in resolved:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

def require_ho_map(raw_label, ho_img, ho_label_to_id, yeo_masks, row_idx, colname, alias_map=None):
    labs = resolve_ho_labels(raw_label, ho_label_to_id, alias_map=alias_map)
    hids = [ho_label_to_id[lab] for lab in labs]

    seed_mask = np.zeros(ho_img.shape[:3], dtype=bool)
    ho_data = ho_img.get_fdata().astype(int)
    for hid in hids:
        seed_mask |= (ho_data == hid)

    yeo, best_d, best_inter = best_yeo_for_mask(seed_mask, yeo_masks, row_idx=row_idx, colname=colname)
    return labs, hids, yeo, best_d, best_inter

def pick_coord_columns(df):
    colmap = {norm(c): c for c in df.columns}
    candidates = [
        ("xtarget", "ytarget", "ztarget"),
        ("x", "y", "z"),
        ("targetx", "targety", "targetz"),
    ]
    for a, b, c in candidates:
        if a in colmap and b in colmap and c in colmap:
            return colmap[a], colmap[b], colmap[c]
    raise ValueError(
        "Could not find target coordinate columns. I looked for variants of "
        "['x (target)', 'y (target)', 'z (target)'] and ['x','y','z']."
    )

# =========================
# Run
# =========================
kaelen = pd.read_excel(KAELEN_XLSX)
kaelen.columns = kaelen.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

x_col, y_col, z_col = pick_coord_columns(kaelen)

ho_label_to_id = load_ho_label_to_id(HO_TXT)
ho_to_yeo, ho_debug, ho_img = compute_ho_to_yeo7(HO_NII, YEO_NII)
yeo_masks = get_yeo_masks(ho_img, YEO_NII)

seed_out, targ_out = [], []
seed_labs_debug, seed_hids_debug = [], []
seed_dice_debug, seed_inter_debug = [], []
targ_dice_debug, targ_inter_debug = [], []
debug_rows = []

for i, r in kaelen.iterrows():
    seed_labels, seed_hids, seed_yeo, seed_dice, seed_inter = require_ho_map(
        r["Seed"], ho_img, ho_label_to_id, yeo_masks, i, "Seed", alias_map=KAELEN_SEED_ALIAS
    )

    coord = (float(r[x_col]), float(r[y_col]), float(r[z_col]))
    targ_mask = sphere_mask_on_grid(ho_img, coord, radius_mm=TARGET_RADIUS_MM)
    targ_yeo, targ_dice, targ_inter = best_yeo_for_mask(targ_mask, yeo_masks, row_idx=i, colname="Target")

    seed_out.append(seed_yeo)
    targ_out.append(targ_yeo)

    seed_labs_debug.append(" | ".join(seed_labels))
    seed_hids_debug.append(" | ".join(map(str, seed_hids)))
    seed_dice_debug.append(seed_dice)
    seed_inter_debug.append(seed_inter)
    targ_dice_debug.append(targ_dice)
    targ_inter_debug.append(targ_inter)

    debug_rows.append({
        "Row": i,
        "Seed_raw": r["Seed"],
        "Seed_HO_labels": " | ".join(seed_labels),
        "Seed_HO_IDs": " | ".join(map(str, seed_hids)),
        "Seed_Yeo7": seed_yeo,
        "Seed_Dice": seed_dice,
        "Seed_Intersect_Voxels": seed_inter,
        "Target_xyz": coord,
        "Target_Yeo7": targ_yeo,
        "Target_Dice": targ_dice,
        "Target_Intersect_Voxels": targ_inter,
    })

kaelen["Seed_Yeo7"] = seed_out
kaelen["Target_Yeo7"] = targ_out
kaelen["Seed_HO_labels"] = seed_labs_debug
kaelen["Seed_HO_IDs"] = seed_hids_debug
kaelen["Seed_Yeo7_Dice"] = seed_dice_debug
kaelen["Seed_Yeo7_Intersect_Voxels"] = seed_inter_debug
kaelen["Target_Yeo7_Dice"] = targ_dice_debug
kaelen["Target_Yeo7_Intersect_Voxels"] = targ_inter_debug

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    kaelen.to_excel(writer, index=False, sheet_name="Kaelen_2016_Yeo7")
    pd.DataFrame(debug_rows).to_excel(writer, index=False, sheet_name="Debug_rowwise")
    ho_debug.to_excel(writer, index=False, sheet_name="Debug_HO_to_Yeo7")

print(f"Saved: {OUT_XLSX}")
print("\nRow-wise debug preview:")
print(pd.DataFrame(debug_rows).head(10).to_string(index=False))


Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_yeo_converted/Kaelen_2016_Yeo7_converted.xlsx

Row-wise debug preview:
 Row Seed_raw    Seed_HO_labels Seed_HO_IDs Seed_Yeo7  Seed_Dice  Seed_Intersect_Voxels          Target_xyz    Target_Yeo7  Target_Dice  Target_Intersect_Voxels
   0      PHC pPaHC l | pPaHC r     65 | 64    Visual   0.040144                   1433   (8.0, -76.0, 8.0)         Visual     0.041137                     1437
   1      PHC pPaHC l | pPaHC r     65 | 64    Visual   0.040144                   1433   (6.0, -84.0, 8.0)         Visual     0.032377                     1131
   2      PHC pPaHC l | pPaHC r     65 | 64    Visual   0.040144                   1433   (4.0, -80.0, 6.0)         Visual     0.040794                     1425
   3      PHC pPaHC l | pPaHC r     65 | 64    Visual   0.040144                   1433 (-8.0, -88.0, 10.0)         Visual     0.022615                      790
   4      PHC pPaHC l | pPaHC

In [2]:
# =========================
# CONVERTING Gaddis 2022 (Smith 2009 seed + Smith 2009 target) -> Yeo-7
#   FIXED: threshold Smith component maps before Dice
# =========================
import os, re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

GADDIS_XLSX  = os.path.join(BASE_DIR, "Gaddis_2022_significant_edges_FINAL.xlsx")
SMITH_NII    = os.path.join(ATLAS_DIR, "PNAS_Smith09_rsn10.nii.gz")
SMITH_LABELS = os.path.join(ATLAS_DIR, "SMITH09_RSN10_labels.txt")
YEO_NII      = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR  = os.path.join(BASE_DIR, "data_yeo_converted")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Gaddis_2022_Yeo7_converted.xlsx")

YEO7 = ["Visual", "Somatomotor", "DorsalAttention", "VentralAttention", "Limbic", "Frontoparietal", "Default"]
SMITH_Z_THRESHOLD = 2.3

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())


def norm(s):
    s = clean_ws(s).lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s


def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)


# =========================
# Load Smith RSN10 canonical labels
# =========================
def load_smith_labels(labels_path):
    id_to_label = {}
    with open(labels_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            m = re.match(r"^(\d+)\s+(.+)$", ln)
            if not m:
                continue
            rid = int(m.group(1))
            lab = clean_ws(m.group(2))
            id_to_label[rid] = lab
    label_to_id = {norm(v): k for k, v in id_to_label.items()}
    return id_to_label, label_to_id


# =========================
# Label normalization for Gaddis
# =========================
SMITH_ALIAS = {
    # Canonical/common
    "cerebellum": "Cerebellum",
    "cerebellar": "Cerebellum",
    "auditory": "Auditory",
    "aud": "Auditory",
    "sensorimotor": "Sensorimotor",
    "somatomotor": "Sensorimotor",
    "sm": "Sensorimotor",
    "defaultmode": "Default mode network",
    "defaultmodenetwork": "Default mode network",
    "dmn": "Default mode network",
    "executive": "Executive control",
    "executivecontrol": "Executive control",
    "ecn": "Executive control",
    "leftfrontoparietal": "Left frontoparietal",
    "lfp": "Left frontoparietal",
    "frontoparietal1": "Left frontoparietal",
    "rightfrontoparietal": "Right frontoparietal",
    "rfp": "Right frontoparietal",
    "frontoparietal2": "Right frontoparietal",
    "visualmedial": "Visual medial",
    "visualoccipitalpole": "Visual occipital pole",
    "visuallateral": "Visual lateral",
    "vism": "Visual medial",
    "viso": "Visual occipital pole",
    "visl": "Visual lateral",

    # Gaddis spreadsheet forms with parentheses/spaces
    "visualmedial": "Visual medial",
    "visualoccipitalpole": "Visual occipital pole",
    "visuallateral": "Visual lateral",
}


# =========================
# Smith RSN10 -> Yeo-7 via Dice
# =========================
def compute_smith_to_yeo7(smith_nii_path, yeo_nii_path, zthr=2.3):
    smith_img = nib.load(smith_nii_path)
    yeo_img = nib.load(yeo_nii_path)

    smith = smith_img.get_fdata()
    if smith.ndim != 4:
        raise ValueError(f"Expected Smith atlas to be 4D. Got shape: {smith.shape}")
    if smith.shape[-1] < 10:
        raise ValueError(f"Expected >=10 Smith components. Got: {smith.shape[-1]}")

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, smith_img, interpolation="nearest")
    yeo = yeo_rs.get_fdata().astype(int)

    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    smith_to_yeo = {}
    debug_rows = []

    for k in range(10):
        comp = smith[..., k]
        comp_mask = comp > zthr

        if np.count_nonzero(comp_mask) == 0:
            raise ValueError(
                f"Smith component {k + 1} has zero voxels above threshold {zthr}. "
                f"Try lowering SMITH_Z_THRESHOLD (e.g., 2.0 or 1.5)."
            )

        best_name, best_d, best_inter = None, -1.0, -1
        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(comp_mask & ym)
            d = dice(comp_mask, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)

        smith_to_yeo[k + 1] = best_name
        debug_rows.append({
            "Smith_Component": k + 1,
            "Best_Yeo7": best_name,
            "Dice": best_d,
            "Intersect_Voxels": best_inter,
            "Component_Voxels": int(np.count_nonzero(comp_mask)),
        })

    debug_df = pd.DataFrame(debug_rows)
    print("\nSmith -> Yeo-7 mapping:")
    print(debug_df.to_string(index=False))
    return smith_to_yeo


# =========================
# Convert Gaddis labels to canonical Smith labels
# =========================
def canonical_label(x, row_idx, colname, id_to_label):
    raw = clean_ws(x)
    k = norm(raw)

    for rid, lab in id_to_label.items():
        if norm(lab) == k:
            return lab

    if k in SMITH_ALIAS:
        canon = SMITH_ALIAS[k]
        for rid, lab in id_to_label.items():
            if norm(lab) == norm(canon):
                return lab

    hints = [lab for rid, lab in id_to_label.items() if k in norm(lab) or norm(lab) in k][:10]
    hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints) if hints else ""
    raise ValueError(
        f"[Gaddis] Row {row_idx}: could not map '{colname}' label to Smith RSN10.\n"
        f"  Raw value: {raw!r}\n"
        f"  Normalized: {k!r}{hint_txt}"
    )


# =========================
# Run
# =========================
id_to_label, label_to_id = load_smith_labels(SMITH_LABELS)
smith_to_yeo = compute_smith_to_yeo7(SMITH_NII, YEO_NII, zthr=SMITH_Z_THRESHOLD)

gaddis = pd.read_excel(GADDIS_XLSX)
gaddis.columns = gaddis.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

if "Network" in gaddis.columns:
    net_norm = (
        gaddis["Network"].astype(str).str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "", regex=True)
    )
    sub = gaddis[net_norm.isin(["smith2009", "smith", "smithrsn10"])].copy()
    if len(sub) == 0:
        sub = gaddis.copy()
else:
    sub = gaddis.copy()

seed_out, targ_out = [], []
debug_rows = []

for i, r in sub.iterrows():
    seed_lab = canonical_label(r["Seed"], i, "Seed", id_to_label)
    targ_lab = canonical_label(r["Target"], i, "Target", id_to_label)

    sid = label_to_id[norm(seed_lab)]
    tid = label_to_id[norm(targ_lab)]

    s_yeo = smith_to_yeo.get(sid)
    t_yeo = smith_to_yeo.get(tid)

    if s_yeo not in YEO7:
        raise ValueError(f"[Gaddis] Row {i}: Seed {seed_lab!r} mapped to invalid Yeo-7 value: {s_yeo!r}")
    if t_yeo not in YEO7:
        raise ValueError(f"[Gaddis] Row {i}: Target {targ_lab!r} mapped to invalid Yeo-7 value: {t_yeo!r}")

    seed_out.append(s_yeo)
    targ_out.append(t_yeo)

    debug_rows.append({
        "Row": i,
        "Seed_Raw": r["Seed"],
        "Seed_Smith_Label": seed_lab,
        "Seed_Smith_ID": sid,
        "Seed_Yeo7": s_yeo,
        "Target_Raw": r["Target"],
        "Target_Smith_Label": targ_lab,
        "Target_Smith_ID": tid,
        "Target_Yeo7": t_yeo,
    })

out = pd.DataFrame({
    "Study": sub["Study"].astype(str).values,
    "Contrast": sub["Contrast"].astype(str).values,
    "Seed": seed_out,
    "Target": targ_out,
    "Direction": sub["Direction"].astype(str).values,
})

out.to_excel(OUT_XLSX, index=False)
print(f"\n✅ Saved Gaddis: {OUT_XLSX} | Rows: {len(out)}")

print("\nPreview:")
print(out.to_string(index=False))

print("\nGaddis label mapping debug:")
print(pd.DataFrame(debug_rows).to_string(index=False))



Smith -> Yeo-7 mapping:
 Smith_Component        Best_Yeo7     Dice  Intersect_Voxels  Component_Voxels
               1           Visual 0.278647              3464             16664
               2           Visual 0.198226              2090             12888
               3           Visual 0.234508              3894             25011
               4          Default 0.342841              6178             19187
               5           Visual 0.062025               706             14566
               6      Somatomotor 0.244239              4086             24959
               7 VentralAttention 0.217414              3326             23180
               8   Frontoparietal 0.146063              3303             34771
               9   Frontoparietal 0.219659              4152             27348
              10   Frontoparietal 0.183292              3278             25312

✅ Saved Gaddis: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/data_y